# Brand recommender

Generated moodboards with No, thanks, Maybe, and Love it feedback. Rewards are 0, 0.5, and 1.


## Setup

Repo-root or step-folder paths.


In [ ]:
from pathlib import Path

helper_path = Path("helpers/recommender_helpers.py")
if not helper_path.exists():
    helper_path = Path("step_5_recommender/helpers/recommender_helpers.py")
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())


repro_path = Path("helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("step_5_recommender/../helpers/reproducibility_helpers.py")
exec(compile(repro_path.read_text(encoding="utf-8"), str(repro_path), "exec"), globals())
ROOT = find_repo_root(Path.cwd())


## Quick checks

Recommender files and small feedback notes.


In [ ]:
from IPython.display import display

embedding_metadata, embedding_matrix, resolved_embedding_dir = load_embeddings(ROOT)
recommender_artifacts = pd.DataFrame([
    {"artifact": "embedding metadata", "path": str(resolved_embedding_dir / "brand_metadata.csv"), "rows": len(embedding_metadata)},
    {"artifact": "embedding matrix", "path": str(resolved_embedding_dir / "brand_embeddings.npz"), "rows": embedding_matrix.shape[0]},
    {"artifact": "moodboard-backed catalog", "path": str(MOODBOARD_ROOT), "rows": len(catalog)},
])
feedback_table = recommender_feedback_table()

save_table(recommender_artifacts, "recommender_artifacts", ROOT)
save_table(feedback_table, "recommender_user_feedback", ROOT)

display(recommender_artifacts)
display(feedback_table)


## Swipe

Varied moodboards first, then recommendations after enough feedback.


In [ ]:
STATE = start_recommender(category="clothes", batch_size=10, min_loves=2, alpha=0.35, seed=7, top_k=8, max_steps=30)


## Manual test

Fixed feedback lists for a quick ranking check.


In [ ]:

TEST_CATEGORY = "clothes"
manual_catalog = category_catalog(TEST_CATEGORY)
manual_embeddings = category_embeddings(TEST_CATEGORY)
manual_sample = manual_catalog.sample(n=min(10, len(manual_catalog)), random_state=7).copy()
manual_sample[['brand_name', 'category', 'aesthetic_keywords', 'moodboard_path']]


In [ ]:

LOVE_IT_BRANDS = [
    # 'Jil Sander',
]

MAYBE_BRANDS = [
    # 'Toteme',
]

NO_THANKS_BRANDS = [
    # 'Gucci',
]

name_to_idx = {name: i for i, name in enumerate(manual_catalog['brand_name'])}
manual_feedback = [(name_to_idx[name], REWARD_VALUES['Love it']) for name in LOVE_IT_BRANDS if name in name_to_idx]
manual_feedback += [(name_to_idx[name], REWARD_VALUES['Maybe']) for name in MAYBE_BRANDS if name in name_to_idx]
manual_feedback += [(name_to_idx[name], REWARD_VALUES['No, thanks']) for name in NO_THANKS_BRANDS if name in name_to_idx]
manual_seen = [name_to_idx[name] for name in [*LOVE_IT_BRANDS, *MAYBE_BRANDS, *NO_THANKS_BRANDS] if name in name_to_idx]

manual_bandit = fit_bandit(manual_feedback, manual_embeddings, alpha=0.35)
manual_recs = rank_brands(manual_bandit, manual_embeddings, manual_catalog, seen=manual_seen, top_k=12)
manual_recs
